# Decision Calibration — verified continuation

Notebook độc lập: đã nhúng source, 11.000 prediction cũ và gold snapshot, không cần upload thêm file.

**Mục tiêu:** phân tích đúng theo case, chạy thêm OOD cân bằng có rating ordinal, xuất ZIP để gửi lại. Bản thảo/verification report nhúng là bản đang làm, không phải paper đã hoàn tất.

1. Runtime → Change runtime type → chọn T4/L4 GPU.
2. Giữ cấu hình mặc định, chọn Run all. Hai d1 chạy lần lượt; không dùng BF16.
3. Gửi lại `decision_calibration_outputs.zip` tải xuống ở cell cuối.

Lượt mặc định: 5.000 quyết định/model (2.000 typed + 3.000 OOD); hai model mới = 10.000 quyết định. Thời gian phụ thuộc GPU và tải model; không có thời gian chạy GPU đã xác minh cho notebook mới này.

Có thể đặt `RUN_GPU_EXTENSION=False` để chỉ tái phân tích 5 model cũ, không cần GPU. Để giữ tiến độ khi runtime mất kết nối, đặt `USE_DRIVE=True`; nếu không, tải ZIP sau mỗi model/cell export cuối.


In [ ]:
# Cấu hình — giữ nguyên cho lượt nghiên cứu chính
RUN_GPU_EXTENSION = True
MODELS_TO_RUN = ['d1-omni', 'd1-3b']
# Mở rộng cả 5: ['d1-omni','d1-3b','opendecider-small','jeff-2b','jeff-0.8b']
QUOTA_PER_CLASS = 250  # AG News/Rotten/SST-2
YELP_PER_CLASS = 200  # 5 class: 1000 rating decisions
N_SPLIT_SEEDS = 20
N_BOOTSTRAP = 1000
USE_DRIVE = False
OUTPUT_FOLDER_NAME = 'decision_calibration_verified_v2'


In [ ]:
# Dependencies: giữ PyTorch/CUDA có sẵn của Colab; scripts chạy trong subprocess mới.
import subprocess, sys, os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
packages = ['numpy>=1.26,<3', 'scipy>=1.11', 'matplotlib>=3.6', 'pandas>=2.2']
if RUN_GPU_EXTENSION:
    packages += ['transformers>=5.15,<6', 'datasets>=3', 'huggingface_hub>=0.24',
                 'safetensors>=0.4', 'pillow>=10', 'soundfile>=0.12', 'accelerate']
    if 'opendecider-small' in MODELS_TO_RUN:
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
        packages += ['opendecider[small]']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', *packages], check=True)
print('Dependencies installed. Không cần import model vào notebook kernel.')


In [ ]:
# Giải nén payload đã nhúng, không tải code từ một repo không xác định.
from pathlib import Path
import base64, io, zipfile, json, hashlib
ROOT = Path('/content/decision_calibration_source')
ROOT.mkdir(exist_ok=True)
PAYLOAD_B64 = '...'
payload = base64.b64decode(PAYLOAD_B64)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        target = (ROOT / item.filename).resolve()
        assert target.is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive') / OUTPUT_FOLDER_NAME
else:
    OUTPUT = Path('/content') / OUTPUT_FOLDER_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Source:', ROOT, '\nOutput:', OUTPUT)
print('Payload SHA256:', hashlib.sha256(payload).hexdigest())
# Log riêng của từng subprocess, có stderr và exit status, không giấu failure.
def run_logged(args, label):
    logdir = OUTPUT / 'logs'; logdir.mkdir(exist_ok=True)
    logfile = logdir / (label + '.log')
    with logfile.open('a') as log:
        log.write('\nCOMMAND: ' + repr(args) + '\n'); log.flush()
        proc = subprocess.Popen([str(x) for x in args], cwd=ROOT, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end=''); log.write(line); log.flush()
        status = proc.wait(); log.write('\nEXIT CODE: ' + str(status) + '\n')
    if status:
        raise RuntimeError(f'{label} failed ({status}). Xem {logfile}; vẫn chạy cell export cuối.')


## 1. Kiểm tra và phân tích từ predictions đã có — CPU

Chia 80 case fit/320 case eval, giữ toàn bộ câu hỏi của mỗi case. 20 seed và 1000 bootstrap có refit temperature. Không diễn giải spread giữa seed là confidence interval.


In [ ]:
EXISTING_RESULTS = OUTPUT / 'existing_results'
run_logged([sys.executable, ROOT/'scripts/test_verified.py'], 'regression')
run_logged([sys.executable, ROOT/'scripts/verify_analysis.py', '--out', EXISTING_RESULTS,
            '--seeds', N_SPLIT_SEEDS, '--bootstrap', N_BOOTSTRAP], 'existing_analysis')
run_logged([sys.executable, ROOT/'scripts/summarize_verified.py', EXISTING_RESULTS], 'existing_summary')
run_logged([sys.executable, ROOT/'scripts/make_verified_figures.py', EXISTING_RESULTS], 'existing_figures')
from IPython.display import Markdown, display
display(Markdown((EXISTING_RESULTS/'SUMMARY.md').read_text()))


## 2. Inference mới — GPU

Dùng checkpoint snapshot và datasets đã resolve hash, giữ exact input cases. Probe kiểm tra từng domain trước full run. Nếu run thiếu mẫu, script báo lỗi; chạy lại cùng folder/config để tiếp tục. Không đổi quota/pins giữa các model.

Chỉ chạy default hai d1 thì kết luận OOD mới chỉ áp dụng cho hai d1. Để kết luận semantic OOD xuyên họ, chạy cùng cases cho cả 5.


In [ ]:
EXTENSION = OUTPUT / 'extension_output'
FAILURES = {}
COMPLETED = []
if RUN_GPU_EXTENSION:
    result = subprocess.run([sys.executable, '-c',
        "import torch; assert torch.cuda.is_available(), 'Chọn GPU runtime'; print(torch.cuda.get_device_name()); print(torch.__version__)"],
        text=True, capture_output=True)
    print(result.stdout); print(result.stderr)
    assert result.returncode == 0, 'Không có GPU. Chọn T4/L4 hoặc RUN_GPU_EXTENSION=False.'
    # Jeff source relies on the Transformers/Torch stack installed above.
    # First inference resolves and pins its source commit as well as all model/dataset hashes.
    for model_name in MODELS_TO_RUN:
        common = [sys.executable, ROOT/'scripts/run_extension.py', '--model', model_name,
                  '--out', EXTENSION, '--quota', QUOTA_PER_CLASS,
                  '--yelp-per-class', YELP_PER_CLASS, '--include-typed']
        try:
            run_logged(common + ['--probe'], 'probe_'+model_name)
            run_logged(common, 'inference_'+model_name)
            manifest = json.loads((EXTENSION/('manifest_'+model_name+'.json')).read_text())
            assert manifest['complete'] and manifest['actual_rows'] == manifest['expected_rows']
            COMPLETED.append(model_name)
        except Exception as error:
            FAILURES[model_name] = repr(error)
            print('FAILED', model_name, error)
        # Tải checkpoint output sau mỗi model nếu không dùng Drive.
        # Cell export cuối vẫn cung cấp ZIP chung, gồm log cả các model thất bại.
        if not USE_DRIVE:
            import shutil
            from google.colab import files
            checkpoint_zip = shutil.make_archive('/content/checkpoint_'+model_name, 'zip', OUTPUT)
            files.download(checkpoint_zip)
    (OUTPUT/'run_status.json').write_text(json.dumps({'completed': COMPLETED, 'failures': FAILURES}, indent=2))
    print('Complete models:', COMPLETED, '\nFailures:', FAILURES)
else:
    print('GPU extension skipped by configuration.')


## 3. Phân tích mới và raw/shipped arm

Chỉ nhận model có complete manifest. Kết quả Yelp score có hard ordinal RPS/MAE; full typed soft metrics ghi riêng. Không báo per-task transfer sang domain chưa có fitted temperature.


In [ ]:
if RUN_GPU_EXTENSION and COMPLETED:
    NEW_RESULTS = OUTPUT / 'extension_results'
    run_logged([sys.executable, ROOT/'scripts/verify_analysis.py', '--data-dir', EXTENSION,
                '--models', *COMPLETED, '--out', NEW_RESULTS,
                '--seeds', N_SPLIT_SEEDS, '--bootstrap', N_BOOTSTRAP], 'extension_analysis')
    run_logged([sys.executable, ROOT/'scripts/summarize_verified.py', NEW_RESULTS], 'extension_summary')
    run_logged([sys.executable, ROOT/'scripts/make_verified_figures.py', NEW_RESULTS], 'extension_figures')
    if 'd1-omni' in COMPLETED:
        run_logged([sys.executable, ROOT/'scripts/analyze_raw_arm.py', EXTENSION], 'raw_shipped')
    display(Markdown((NEW_RESULTS/'SUMMARY.md').read_text()))
else:
    print('Chưa có complete inference mới; xem logs rồi gửi ZIP hoặc rerun.')


## 4. Export — gửi lại ZIP này

Chạy cell này kể cả khi inference bị lỗi để giữ logs. ZIP không chứa model weights/token/API key. Source/input JSONL và manifest chứa dữ liệu benchmark public. Nếu dùng Drive, output còn trong folder đã chọn.


In [ ]:
import shutil, importlib.metadata
from google.colab import files
# Giữ exact source đã chạy để đối chiếu code với output.
source_copy = OUTPUT/'executed_source'
source_copy.mkdir(exist_ok=True)
for sub in ['decision_calib','scripts']:
    shutil.copytree(ROOT/sub, source_copy/sub, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
for name in ['VERIFY_REPORT.md','PAPER_DRAFT.md','README.md']:
    shutil.copy2(ROOT/name, source_copy/name)
versions = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(OUTPUT/'pip_freeze.txt').write_text(versions)
(OUTPUT/'notebook_config.json').write_text(json.dumps({
    'RUN_GPU_EXTENSION': RUN_GPU_EXTENSION, 'MODELS_TO_RUN': MODELS_TO_RUN,
    'QUOTA_PER_CLASS': QUOTA_PER_CLASS, 'YELP_PER_CLASS': YELP_PER_CLASS,
    'N_SPLIT_SEEDS': N_SPLIT_SEEDS, 'N_BOOTSTRAP': N_BOOTSTRAP,
    'USE_DRIVE': USE_DRIVE, 'payload_sha256': hashlib.sha256(payload).hexdigest()}, indent=2))
archive = shutil.make_archive('/content/decision_calibration_outputs', 'zip', OUTPUT)
print('Gửi lại file:', archive)
files.download(archive)
